# SAM 3D Objects - Colab Setup

This notebook sets up the environment and runs a demo inference on Google Colab (T4 GPU recommended).

**Note:** If you encounter build errors with PyTorch3D or other libraries, make sure you are using a GPU runtime.

In [ ]:
# 1. Clone the repository
import os
if not os.path.exists("sam-3d-objects"):
    !git clone https://github.com/facebookresearch/sam-3d-objects.git
    %cd sam-3d-objects
else:
    %cd sam-3d-objects
    !git pull

# 2. Install Dependencies
print("Installing dependencies... This may take a few minutes.")

# Install general requirements first (excluding pytorch3d)
!pip install -r requirements_colab.txt

# 3. Install PyTorch3D (Binary Wheel)
# Building PyTorch3D from source on Colab is slow and error-prone. We use pre-built wheels.
import torch
import sys

try:
    import pytorch3d
    print("PyTorch3D is already installed.")
except ImportError:
    print("Installing PyTorch3D...")
    pyt_version_str = torch.__version__.split("+")[0].replace(".", "")
    cuda_version_str = torch.version.cuda.replace(".", "")
    
    # Handle different CUDA versions if needed, but generic URL construction usually works for major versions
    # Fallback/Default for Colab (usually PyTorch 2.x + CUDA 12.x)
    # Facebook provides wheels for specific combinations. 
    # Since official wheels might lag behind latest PyTorch/CUDA on Colab, we try a best-effort approach.
    
    if "2.4" in torch.__version__ or "2.5" in torch.__version__:
        # For very new PyTorch, official wheels might be missing. 
        # We try to install from the nightly or specific feed, or fall back to git if needed.
        # But first, let's try the standard pattern
        try:
             !pip install --no-index --no-cache-dir pytorch3d -f https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/py310_cu121_pyt240/download.html
        except:
             print("Binary wheel not found, falling back to source install (slow)...")
             !pip install "git+https://github.com/facebookresearch/pytorch3d.git@stable"
    else:
         # Attempt to construct URL
         version_str = "".join([
             f"py3{sys.version_info.minor}_cu",
             cuda_version_str,
             f"_pyt{pyt_version_str}"
         ])
         !pip install --no-index --no-cache-dir pytorch3d -f https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/{version_str}/download.html

# 4. Install the package in editable mode and patch Hydra
!pip install -e .
!python patching/hydra

print("Installation complete. PLEASE RESTART THE RUNTIME (Runtime > Restart session) if you see import errors!")

## Download Checkpoints
You need a Hugging Face token to download the model weights. Accept the license at https://huggingface.co/facebook/sam-3d-objects first.

In [ ]:
from huggingface_hub import snapshot_download
import os
import shutil

# Function to download weights
def download_weights(token):
    tag = "hf"
    download_dir = f"checkpoints/{tag}-download"
    target_dir = f"checkpoints/{tag}"
    
    if os.path.exists(target_dir) and os.path.exists(os.path.join(target_dir, "pipeline.yaml")):
        print(f"Checkpoints already exist at {target_dir}")
        return
        
    print("Downloading model weights...")
    try:
        snapshot_download(
            repo_id="facebook/sam-3d-objects",
            repo_type="model",
            local_dir=download_dir,
            max_workers=1,
            token=token
        )
        
        # Move checkpoints to correct location
        source = os.path.join(download_dir, "checkpoints")
        if os.path.exists(source):
            if os.path.exists(target_dir):
                shutil.rmtree(target_dir)
            shutil.move(source, target_dir)
            shutil.rmtree(download_dir)
            print("Download complete and files moved.")
        else:
             # Fallback if structure is different
            if os.path.exists(target_dir):
                shutil.rmtree(target_dir)
            shutil.move(download_dir, target_dir)
            print("Download complete (fallback structure).")
            
    except Exception as e:
        print(f"Error downloading weights: {e}")

# Input your HF token here
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except:
    token = None

if not token:
    print("Please enter your Hugging Face Token (input hidden):")
    from getpass import getpass
    token = getpass()

download_weights(token)

In [ ]:
# Minimal Inference Code
import sys
import os
sys.path.append("notebook")
from inference import Inference, load_image, load_single_mask
import torch

if not torch.cuda.is_available():
    print("Warning: CUDA is not available. Inference will be slow or fail.")

tag = "hf"
config_path = f"checkpoints/{tag}/pipeline.yaml"

if not os.path.exists(config_path):
    print(f"Error: Config not found at {config_path}. Did you download weights?")
else:
    # Load model
    print("Loading model...")
    inference = Inference(config_path, compile=False)

    # Load dummy image/mask (using one from repo)
    image_path = "notebook/images/shutterstock_stylish_kidsroom_1640806567/image.png"
    mask_folder = "notebook/images/shutterstock_stylish_kidsroom_1640806567"
    
    print(f"Processing image: {image_path}")
    if os.path.exists(image_path):
        image = load_image(image_path)
        mask = load_single_mask(mask_folder, index=14)

        # Run model
        output = inference(image, mask, seed=42)

        # Save output
        output["gs"].save_ply("splat.ply")
        print("Success! Output saved to splat.ply")
    else:
        print("Test image not found.")